# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [3]:
# 1. Check the dimensions of the dataframe (Rows, Columns)
shape = spaceship.shape
print(f"Data shape: {shape}")


Data shape: (8693, 14)


**Check for data types**

In [4]:
# 2. Check the data type of each column to understand the content
data_types = spaceship.dtypes
print("\nData types:")
print(data_types)


Data types:
PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object


**Check for missing values**

In [9]:
spaceship.isnull().any()

PassengerId     False
HomePlanet       True
CryoSleep        True
Cabin            True
Destination      True
Age              True
VIP              True
RoomService      True
FoodCourt        True
ShoppingMall     True
Spa              True
VRDeck           True
Name             True
Transported     False
dtype: bool

In [10]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [11]:
# 1. Drop rows with any missing values
spaceship.dropna(inplace=True)


- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [13]:
list(spaceship['Cabin'].unique())

['B', 'F', 'A', 'G', 'E', 'C', 'D', 'T']

In [14]:
spaceship['Cabin'] = spaceship['Cabin'].apply(lambda x: x.split("/")[0])

In [15]:
spaceship['Cabin']

0       B
1       F
2       A
3       A
4       F
       ..
8688    A
8689    G
8690    G
8691    E
8692    E
Name: Cabin, Length: 6606, dtype: object

- Drop PassengerId and Name

In [16]:
# Remove columns that do not provide predictive patterns
spaceship.drop(['PassengerId', 'Name'], axis=1, inplace=True)

# Show the first few rows to see the cleaner data
print(spaceship.head())

  HomePlanet CryoSleep Cabin  Destination   Age    VIP  RoomService  \
0     Europa     False     B  TRAPPIST-1e  39.0  False          0.0   
1      Earth     False     F  TRAPPIST-1e  24.0  False        109.0   
2     Europa     False     A  TRAPPIST-1e  58.0   True         43.0   
3     Europa     False     A  TRAPPIST-1e  33.0  False          0.0   
4      Earth     False     F  TRAPPIST-1e  16.0  False        303.0   

   FoodCourt  ShoppingMall     Spa  VRDeck  Transported  
0        0.0           0.0     0.0     0.0        False  
1        9.0          25.0   549.0    44.0         True  
2     3576.0           0.0  6715.0    49.0        False  
3     1283.0         371.0  3329.0   193.0        False  
4       70.0         151.0   565.0     2.0         True  


- For non-numerical columns, do dummies.

In [17]:
for column in spaceship.select_dtypes('object'):
    if len(spaceship[column].unique()) > 2:
        dummies = pd.get_dummies(spaceship[column], dtype=int, prefix=column)
    else:
        dummies = pd.get_dummies(spaceship[column], drop_first=True, dtype=int, prefix=column)
    spaceship = pd.concat([spaceship.drop(columns=[column]), dummies], axis=1)

In [18]:
spaceship['Transported'] = spaceship['Transported'].astype(int)

**Perform Train Test Split**

In [19]:
spaceship

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,...,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
1,24.0,109.0,9.0,25.0,549.0,44.0,1,1,0,0,...,0,0,0,1,0,0,0,0,1,0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,1
3,33.0,0.0,1283.0,371.0,3329.0,193.0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
4,16.0,303.0,70.0,151.0,565.0,2.0,1,1,0,0,...,0,0,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,41.0,0.0,6819.0,0.0,1643.0,74.0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1
8689,18.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0,...,0,0,0,0,1,0,0,1,0,0
8690,26.0,0.0,0.0,1872.0,1.0,0.0,1,1,0,0,...,0,0,0,0,1,0,0,0,1,0
8691,32.0,0.0,1049.0,0.0,353.0,3235.0,0,0,1,0,...,0,0,1,0,0,0,1,0,0,0


In [20]:
X = spaceship.drop(columns=['Transported'])

In [21]:
y = spaceship['Transported']

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42, test_size = 0.20)

**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [24]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1. Initialize the model (using 5 neighbors as a default)
knn = KNeighborsClassifier(n_neighbors=5)

# 2. Train the model with our training data
knn.fit(X_train, y_train)

# 3. Make predictions on the test set
y_pred = knn.predict(X_test)

# 4. Evaluate the performance
accuracy = accuracy_score(y_test, y_pred)
print(f"The model accuracy is: {accuracy:.2%}")

The model accuracy is: 78.82%


- Evaluate your model's performance. Comment it

In [27]:
# Calculate accuracy on the test set
accuracy = knn.score(X_test, y_test)
print(f"Model Accuracy: {accuracy:.4f}")

# Optional: Compare with training accuracy to check for overfitting
train_accuracy = knn.score(X_train, y_train)
print(f"Training Accuracy: {train_accuracy:.4f}")

Model Accuracy: 0.7882
Training Accuracy: 0.8259
